# Edu Nexus – Phase 1: File Scanner & Metadata Index

This notebook scans the local edu_nexus_db raw data folders
and builds a metadata index of all available files.

Responsibilities:
- Detect PDFs, PPTs, images, and other files
- Store file paths, types, and source category
- Output a structured file_index.json for downstream pipelines

No content extraction is done in this notebook.


In [12]:
# Standard library imports
import os
import json
from pathlib import Path
from datetime import datetime


In [13]:
# ===== CONFIGURATION =====

# Change this ONLY if folder location changes
BASE_DB_PATH = Path("../../edu_nexus_db")

RAW_PATH = BASE_DB_PATH / "raw"
METADATA_PATH = BASE_DB_PATH / "metadata"

# Output file
INDEX_FILE = METADATA_PATH / "file_index.json"

print("Base DB Path:", BASE_DB_PATH.resolve())
print("Raw Data Path:", RAW_PATH.resolve())


Base DB Path: C:\Users\kulva\Desktop\Minor Project\edu_nexus_db
Raw Data Path: C:\Users\kulva\Desktop\Minor Project\edu_nexus_db\raw


In [14]:
# ===== VALIDATION =====

required_folders = [
    RAW_PATH / "pdf",
    RAW_PATH / "ppt",
    RAW_PATH / "images",
    RAW_PATH / "other",
    METADATA_PATH
]

missing_folders = [str(p) for p in required_folders if not p.exists()]

if missing_folders:
    raise FileNotFoundError(f"Missing required folders: {missing_folders}")

print("All required folders are present ✅")


All required folders are present ✅


In [15]:
# ===== FILE SCANNER =====

SUPPORTED_TYPES = {
    "pdf": [".pdf"],
    "ppt": [".ppt", ".pptx"],
    "images": [".png", ".jpg", ".jpeg", ".tiff"],
    "other": [".txt", ".docx"]
}

file_index = []
file_id = 1

for category, extensions in SUPPORTED_TYPES.items():
    category_path = RAW_PATH / category
    
    for root, _, files in os.walk(category_path):
        for file in files:
            file_path = Path(root) / file
            ext = file_path.suffix.lower()
            
            if ext in extensions:
                file_index.append({
                    "id": file_id,
                    "filename": file_path.name,
                    "extension": ext,
                    "category": category,
                    "absolute_path": str(file_path.resolve()),
                    "status": "raw",
                    "added_on": datetime.utcnow().isoformat()
                })
                file_id += 1

print(f"Total files indexed: {len(file_index)}")


Total files indexed: 3


C:\Users\kulva\AppData\Local\Temp\ipykernel_4332\1132160011.py:29: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "added_on": datetime.utcnow().isoformat()


In [16]:
# ===== SAVE METADATA INDEX =====

METADATA_PATH.mkdir(parents=True, exist_ok=True)

with open(INDEX_FILE, "w", encoding="utf-8") as f:
    json.dump(file_index, f, indent=4)

print(f"Metadata index saved to: {INDEX_FILE.resolve()}")


Metadata index saved to: C:\Users\kulva\Desktop\Minor Project\edu_nexus_db\metadata\file_index.json


In [17]:
# ===== FILE CLASSIFICATION (LOCAL vs OCR) =====

def classify_file(entry):
    """
    Decide how a file should be processed.
    This is a heuristic-based classifier (fast & safe).
    """
    ext = entry["extension"]
    category = entry["category"]

    # Images ALWAYS need OCR
    if category == "images":
        return {
            "processing_route": "ocr_image",
            "reason": "Image file"
        }

    # PDFs & PPTs: assume local first
    if category in ["pdf", "ppt"]:
        return {
            "processing_route": "local_extract",
            "reason": "Attempt local text extraction first"
        }

    # Other text formats (txt, docx)
    return {
        "processing_route": "local_text",
        "reason": "Plain text format"
    }


In [18]:


for entry in file_index:
    classification = classify_file(entry)
    entry.update(classification)

print("File classification completed ✅")


File classification completed ✅


In [19]:
#Save Updated Metadata

with open(INDEX_FILE, "w", encoding="utf-8") as f:
    json.dump(file_index, f, indent=4)

print("Updated metadata index saved with processing routes ✅")


Updated metadata index saved with processing routes ✅
